In [1]:
import pandas as pd
import numpy as np
import xgboost as xgb
from sklearn.metrics import mean_squared_error

In [2]:
df = pd.read_csv("feature_engineered_dataset.csv")
df.head()

,date,store_id,product_id,category,region,inventory_level,units_sold,units_ordered,demand_forecast,price,...,competitor_gap,discounted_price,revenue,conversion_rate,price_demand_ratio,stock_remaining,traffic_intensity,day_of_week,month,is_weekend
0,2022-01-01,S001,P0001,Groceries,North,231,127,55,135.47,33.50,...,3.81,0.00,0.00,0.453571,0.261719,104,1.202586,5,1,1
1,2022-01-01,S001,P0002,Toys,South,204,150,66,144.04,63.01,...,-3.15,0.00,0.00,0.394737,0.417285,54,1.848780,5,1,1
2,2022-01-01,S001,P0003,Toys,West,102,65,51,74.02,27.99,...,-3.33,0.00,0.00,0.312500,0.424091,37,2.009709,5,1,1
3,2022-01-01,S001,P0004,Toys,North,469,61,164,62.18,32.72,...,-2.02,0.00,0.00,0.484127,0.527742,408,0.265957,5,1,1
4,2022-01-01,S001,P0005,Electronics,East,166,14,135,9.26,73.64,...,4.69,73.64,1030.96,0.081871,4.909333,152,1.017964,5,1,1


In [3]:
df = df.sort_values("date")

In [4]:
train = df[df["date"] < "2022-10-01"]
test = df[df["date"] >= "2022-10-01"]

In [5]:
X_train = train.drop(["units_sold", "date"], axis=1)
y_train = train["units_sold"]

X_test = test.drop(["units_sold", "date"], axis=1)
y_test = test["units_sold"]

In [6]:
X_train = pd.get_dummies(X_train, drop_first=True)
X_test = pd.get_dummies(X_test, drop_first=True)

In [7]:
X_train, X_test = X_train.align(X_test, join='left', axis=1, fill_value=0)

In [8]:
model = xgb.XGBRegressor(
    n_estimators=300,
    learning_rate=0.05,
    max_depth=8,
    subsample=0.8,
    colsample_bytree=0.8
)

model.fit(X_train, y_train)

,"objective objective: typing.Union[str, xgboost.sklearn._SklObjWProto, typing.Callable[[typing.Any, typing.Any], typing.Tuple[numpy.ndarray, numpy.ndarray]], NoneType]Specify the learning task and the corresponding learning objective or a customobjective function to be used.For custom objective, see :doc:`/tutorials/custom_metric_obj` and:ref:`custom-obj-metric` for more information, along with the end note forfunction signatures.",'reg:squarederror'
,"base_score base_score: typing.Union[float, typing.List[float], NoneType]The initial prediction score of all instances, global bias.",None
,booster,None
,"callbacks callbacks: typing.Optional[typing.List[xgboost.callback.TrainingCallback]]List of callback functions that are applied at end of each iteration.It is possible to use predefined callbacks by using:ref:`Callback API `... note:: States in callback are not preserved during training, which means callback objects can not be reused for multiple training sessions without reinitialization or deepcopy... code-block:: python for params in parameters_grid: # be sure to (re)initialize the callbacks before each run callbacks = [xgb.callback.LearningRateScheduler(custom_rates)] reg = xgboost.XGBRegressor(**params, callbacks=callbacks) reg.fit(X, y)",None
,colsample_bylevel colsample_bylevel: typing.Optional[float]Subsample ratio of columns for each level.,None
,colsample_bynode colsample_bynode: typing.Optional[float]Subsample ratio of columns for each split.,None
,colsample_bytree colsample_bytree: typing.Optional[float]Subsample ratio of columns when constructing each tree.,0.8
,"device device: typing.Optional[str].. versionadded:: 2.0.0Device ordinal, available options are `cpu`, `cuda`, and `gpu`.",None
,"early_stopping_rounds early_stopping_rounds: typing.Optional[int].. versionadded:: 1.6.0- Activates early stopping. Validation metric needs to improve at least once in every **early_stopping_rounds** round(s) to continue training. Requires at least one item in **eval_set** in :py:meth:`fit`.- If early stopping occurs, the model will have two additional attributes: :py:attr:`best_score` and :py:attr:`best_iteration`. These are used by the :py:meth:`predict` and :py:meth:`apply` methods to determine the optimal number of trees during inference. If users want to access the full model (including trees built after early stopping), they can specify the `iteration_range` in these inference methods. In addition, other utilities like model plotting can also use the entire model.- If you prefer to discard the trees after `best_iteration`, consider using the callback function :py:class:`xgboost.callback.EarlyStopping`.- If there's more than one item in **eval_set**, the last entry will be used for early stopping. If there's more than one metric in **eval_metric**, the last metric will be used for early stopping.",None
,enable_categorical enable_categorical: boolSee the same parameter of :py:class:`DMatrix` for details.,False
,"eval_metric eval_metric: typing.Union[str, typing.List[typing.Union[str, typing.Callable]], typing.Callable, NoneType].. versionadded:: 1.6.0Metric used for monitoring the training result and early stopping. It can be astring or list of strings as names of predefined metric in XGBoost (See:doc:`/parameter`), one of the metrics in :py:mod:`sklearn.metrics`, or anyother user defined metric that looks like `sklearn.metrics`.If custom objective is also provided, then custom metric should implement thecorresponding reverse link function.Unlike the `scoring` parameter commonly used in scikit-learn, when a callableobject is provided, it's assumed to be a cost function and by default XGBoostwill minimize the result during early stopping.For advanced usage on Early stopping like directly choosing to maximize insteadof minimize, see :py:obj:`xgboost.callback.EarlyStopping`.See :doc:`/tutorials/custom_metric_obj` and :ref:`custom-obj-metric` for moreinformation... code-block:: python from sklearn.datasets import load_diabetes 

In [9]:
train_columns = X_train.columns

In [10]:
y_pred = model.predict(X_test)

rmse = np.sqrt(mean_squared_error(y_test, y_pred))
print("RMSE:", rmse)

RMSE: 0.9860592315052905


In [11]:
def find_best_price(row):
    prices = [row["price"] * i for i in [0.8, 0.9, 1.0, 1.1, 1.2]]

    best_price = row["price"]
    best_revenue = 0

    for p in prices:
        temp = row.copy()
        temp["price"] = p

        temp_df = pd.DataFrame([temp])
        temp_df = pd.get_dummies(temp_df, drop_first=True)

        temp_df = temp_df.reindex(columns=train_columns, fill_value=0)

        demand = model.predict(temp_df)[0]
        revenue = demand * p

        if revenue > best_revenue:
            best_revenue = revenue
            best_price = p

    return best_price

In [12]:
test = df[df["date"] >= "2022-10-01"].copy()
test["optimal_price"] = test.apply(find_best_price, axis=1)

In [13]:
revenues = []

for i in range(len(test)):
    row = test.iloc[i]
    price = find_best_price(row)
    revenue = price * row["units_sold"]
    revenues.append(revenue)

In [14]:
baseline = test["price"] * test["units_sold"]

In [15]:
ml = sum(revenues)

lift = (ml - baseline.sum()) / baseline.sum() * 100
print("Lift:", lift)

Lift: 19.99999999999971
